In [5]:
import sys
sys.path.append("/cm/shared/apps/spack/cpu/opt/spack/linux-centos8-zen/gcc-8.3.1/spark-3.2.1-bin-hadoop3.2/python/lib/py4j-0.10.9.3-src.zip")
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewProject") \
    .getOrCreate()

print("please work")

please work


In [6]:
df = spark.read.csv(
    "/expanse/lustre/projects/uci157/zalabadleh/AmazonReviews/*.tsv",
    sep="\t",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: string (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: string (nullable = true)

+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|product_category|star_rati

In [38]:
from pyspark.sql.functions import col, length, when ## importing everything we might use
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Imputer
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [8]:
df.select(
    "star_rating",
    "helpful_votes",
    "total_votes",
    "verified_purchase",
    "product_category",
    "review_body"
).where(
    col("star_rating").isNull() |
    col("helpful_votes").isNull() |
    col("total_votes").isNull() |
    col("verified_purchase").isNull() |
    col("product_category").isNull() |
    col("review_body").isNull()
).count()   ## checking the count for null values

8492

In [21]:
df_dropped = df.dropna(subset=[
    "verified_purchase",
    "product_category",
    "review_body"
])
df_dropped = df_dropped.withColumn(
    "star_rating",
    col("star_rating").cast("double")
)

In [23]:
imputer = Imputer(
    inputCols=[
        "star_rating",
        "helpful_votes",
        "total_votes"
    ],
    outputCols=[
        "star_rating_imputed",
        "helpful_votes_imputed",
        "total_votes_imputed"
    ]
)
# gets the means from each of these variables and inputs them in place of null values
imputer_model = imputer.fit(df_dropped)
df_imputed = imputer_model.transform(df_dropped)

In [24]:
df_features = df_imputed.withColumn(
    "review_length",
    length(col("review_body"))
)

df_features = df_features.withColumn(
    "helpful_label",
    when(col("helpful_votes_imputed") > 0, 1).otherwise(0)
)

In [27]:
df_features.select(
    "review_body",
    "review_length",
    "helpful_votes_imputed",
    "helpful_label"#binary yes or no if a label is helpful
).show(5)

+--------------------+-------------+---------------------+-------------+
|         review_body|review_length|helpful_votes_imputed|helpful_label|
+--------------------+-------------+---------------------+-------------+
|this is the first...|          273|                    0|            0|
|I've always been ...|          272|                    0|            0|
|This is a book ab...|          410|                    1|            1|
|This is quite pos...|          211|                    0|            0|
|The story behind ...|          100|                    0|            0|
+--------------------+-------------+---------------------+-------------+
only showing top 5 rows



In [28]:
product_category_idx = StringIndexer(
    inputCol="product_category",
    outputCol="product_category_index",
    handleInvalid="keep"
)

In [30]:
df_indexed = product_category_idx.fit(df_features).transform(df_features)

In [31]:
product_category_encoder = OneHotEncoder(
    inputCol="product_category_index",
    outputCol="product_category_encoded"
)

In [32]:
encoded_model = product_category_encoder.fit(df_indexed)

df_encoded = encoded_model.transform(df_indexed)

In [33]:
df_encoded.select(
    "product_category",
    "product_category_index",
    "product_category_encoded"
).show(10, truncate=False)##check

+----------------+----------------------+------------------------+
|product_category|product_category_index|product_category_encoded|
+----------------+----------------------+------------------------+
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Music           |18.0                  |(38,[18],[1.0])         |
|Video           |27.0                  |(38,[27],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
|Books           |29.0                  |(38,[29],[1.0])         |
+----------------+----------------------+------------------------+
only showing top 10 rows



In [ ]:
##Self note 38 total categories

In [34]:
assem = VectorAssembler(
    inputCols=[
        "star_rating_imputed",
        "helpful_votes_imputed",
        "total_votes_imputed",
        "review_length"
    ],
    outputCol="features_unscaled"
)

In [35]:
df_assembled = assem.transform(df_encoded)

In [36]:
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features"
)

In [37]:
scaler_model = scaler.fit(df_assembled)

df_scaled = scaler_model.transform(df_assembled)

In [44]:
df_sample = df_scaled.sample(fraction=0.01, seed=42)

train_data, test_data = df_sample.randomSplit([0.8, 0.2], seed=42)

In [45]:
dt1 = DecisionTreeClassifier(
    labelCol="helpful_label",
    featuresCol="features",
    maxDepth=5
)

In [46]:
dt_model = dt1.fit(train_data)

In [47]:
print(dt_model.toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_843e84d3205f, depth=1, numNodes=3, numClasses=2, numFeatures=4
  If (feature 1 <= 0.02783472800472023)
   Predict: 0.0
  Else (feature 1 > 0.02783472800472023)
   Predict: 1.0

